# Description of the Material Plugin

For anisotropic materials, we need access to off-diagonal matrix elements; therefore, to correctly add the material plugin to the software, we need to know how it uses material properties to update fields and how to import the effects of off-diagonal terms to fields. Based on Lumerical instruction [1], the Yee cell is used as FDTD to update the fields. Therefore, each field component will updated at different spatial positions. In the time domain, fields update from the following formalism, which included:

$$
\begin{equation*}
(U_i(t_{n+1}) + \chi^{(1)}_{0})E_i(t_{n+1}) + \varrho ^{(2)}_{i}(t_{n+1}) + \varrho ^{(3)}_{i}(t_{n+1}) = V_i(t_{n+1})
\end{equation*}
$$

Some parameters always provided by the plugin like $U_i(t_{n+1})$, $E_i(t_{n+1})$, and $V_i(t_{n+1})$. Also, $\varrho ^{(k)}_{i}(t_{n+1})$ is then defined as: $ \varrho^{(k)}_{i}(t_{n+1}) = P^{(k)}_i(t) / \varepsilon _0$ for $ k = 2,3$.

From the solution of the update equation, we can bring in the off-diagonal elements to the material.
also assumed $\Delta t$ is small enough and then one can approximate $E_i(t_{n+1}) \approx E_i(t_{n})$, therefore:

$$
\begin{equation*}
E_i(t_{n+1}) = \frac{V_i(t_{n+1}) - \varrho ^{(2)}_{i}(t_{n+1}) - \varrho ^{(3)}_{i} (t_{n+1})}{U_i(t_{n+1}) + \chi^{(1)}_{0}}
\end{equation*}
$$

which:

$$
\begin{equation*} 
\begin{bmatrix}
\varrho ^{(2)}_{x}(t_{n+1}) \\
\varrho ^{(2)}_{y}(t_{n+1}) \\
\varrho ^{(2)}_{z}(t_{n+1})
\end{bmatrix}
=2\begin{bmatrix}
d_{11} & d_{12} & d_{13} & d_{14} & d_{15} & d_{16}\\
d_{21} & d_{22} & d_{23} & d_{24} & d_{25} & d_{26}\\
d_{31} & d_{32} & d_{33} & d_{34} & d_{35} & d_{36}
\end{bmatrix} 
\begin{bmatrix}
E^{2}_{x}(t_{n}) \\
E^{2}_{y}(t_{n}) \\
E^{2}_{z}(t_{n}) \\
2E_{y}(t_{n})E_{z}(t_{n})\\
2E_{x}(t_{n})E_{z}(t_{n})\\
2E_{x}(t_{n})E_{y}(t_{n}) 
\end{bmatrix} 
\end{equation*}
$$

also:

$$
\begin{equation*} 
\begin{bmatrix}
\varrho ^{(3)}_{x}(t_{n+1}) \\
\varrho ^{(3)}_{y}(t_{n+1}) \\
\varrho ^{(3)}_{z}(t_{n+1})
\end{bmatrix}
=2\begin{bmatrix}
T_{11} & T_{12} & T_{13} & T_{14} & T_{15} & T_{16} & T_{17} & T_{18} & T_{19} & T_{110}\\
T_{21} & T_{22} & T_{23} & T_{24} & T_{25} & T_{26} & T_{27} & T_{28} & T_{29} & T_{210} \\
T_{31} & T_{32} & T_{33} & T_{34} & T_{35} & T_{36} & T_{37} & T_{38} & T_{39} & T_{310}
\end{bmatrix} 
\begin{bmatrix}
E^{3}_{x}(t_{n}) \\
E^{3}_{y}(t_{n}) \\
E^{3}_{z}(t_{n}) \\
3E^{2}_{z}(t_{n})E_{x}(t_{n}) \\
3E^{2}_{z}(t_{n})E_{y}(t_{n}) \\
3E^{2}_{y}(t_{n})E_{z}(t_{n}) \\
3E^{2}_{y}(t_{n})E_{x}(t_{n}) \\
3E^{2}_{x}(t_{n})E_{y}(t_{n}) \\
3E^{2}_{x}(t_{n})E_{z}(t_{n}) \\
6E_{x}(t_{n})E_{y}(t_{n})E_{z}(t_{n})
\end{bmatrix} 
\end{equation*}
$$

In order to calculate E fields, the components update in order of Ex, Ey, and then Ez. In the regular plugin framework, each Cartesian component is updated
independently. For example, we know nothing about Ey and Ez when updating Ex at a particular location. Each field update happens in a different C++ function (calculateEx, calculateEy, and calculateEz) [1].

### Algorithm: Lumerical Plugin Algorithm for CalculateEx Function

**Inputs:**  
- $T_{ij}$ and $d_{ij}$  

**Initialization:**  
- $E_i(t_{n+1})$, $\varrho ^{(k)}_{i}(t_{n+1})$, and $\varrho ^{(m)}_{l}(t_{n+1})$ are initialized to zero values and have the same size.  

**Steps:**  
1. For $l$ from 1 to 3:  
    - Calculate $\varrho ^{(2,3)}_{l}(t_{n+1})$.  
2. Update $\varrho ^{(k)}_{i}(t_{n+1})$ as $\varrho ^{(m)}_{l}(t_{n+1})$.  
3. Calculate $E_x(t_{n+1})$.  
4. If $\varrho ^{(k)}_{i}(t_{n+1})$ is fully filled:  
    - Proceed to calculateEy and calculateEz.  


[1]  “MATERIAL PLUGIN: Full tensorial frequency independent chi2”, LUMERICAL INC.,  (May-2018).